# 06 — Explainability

This notebook surfaces *why* each model made the predictions it did. We use
two lightweight, defensible techniques:

1. **TF-IDF + LR coefficient inspection** — directly reads the linear
   coefficients to show which n-grams pulled each class.
2. **Weak-label category attribution** — for each campaign, which of the
   four manipulation registers (sympathy / urgency / guilt / fear) fired
   most strongly?

A full SHAP / attention attribution on the DistilBERT model is computationally
expensive for long MDCC text on CPU and is left to future work (see §10 of
the final report).


In [1]:
# Path-setup boilerplate so the notebook can import src.*
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


project root: /Users/spandarayamajhi/Desktop/Artificial Intelligence Coursework (Final Submission)


In [2]:
import joblib, pandas as pd
from src import config
from src.explainability import top_features_per_class


## 1. Top TF-IDF + LR features

In [3]:
pipe = joblib.load(config.MODELS_DIR / 'tfidf_lr.joblib')
top = top_features_per_class(pipe, n=25)
top


,manipulative_top,manipulative_weight,non_manipulative_top,non_manipulative_weight
0,funeral,21.1072,to help,-1.3907
1,please help,17.7108,bills,-1.3712
2,right now,12.6039,medical expenses,-1.3327
3,immediately,9.1863,please donate,-1.2258
4,death,8.6054,consider,-1.1901
5,right,8.2127,is currently,-1.1645
6,funeral expenses,7.5815,fire,-1.1576
7,please,7.0269,please consider,-1.1560
8,the funeral,5.4689,recovery,-1.1421
9,heartbreaking,5.0594,can help,-0.9969


**Observation.** `please help` is a top *manipulative* feature; `please donate`
is a top *non-manipulative* feature. The linear model has learned a surface
lexical distinction but cannot tell that the two phrases differ mainly in
register, not intent. This is exactly the failure mode that motivates a
contextual model — and exactly the failure mode that the small weak-label
fine-tune of DistilBERT did not fix here.


## 2. Per-campaign category attribution

In [4]:
labelled = pd.read_csv(config.WEAK_LABELS_CSV)
cols = [c for c in labelled.columns if c.startswith('score_')]
labelled['dominant_category'] = labelled[cols].idxmax(axis=1).str.replace('score_','',regex=False)
labelled[['category','binary_label','fine_label','dominant_category']].sample(8, random_state=42)


,category,binary_label,fine_label,dominant_category
12667,Memorial,manipulative,guilt_or_fear,fear_appeal
5369,Animals,manipulative,sympathy_exploitation,sympathy_exploitation
3059,Medical,non_manipulative,non_manipulative,sympathy_exploitation
9754,Memorial,manipulative,sympathy_exploitation,sympathy_exploitation
11941,Memorial,manipulative,guilt_or_fear,fear_appeal
9799,Memorial,manipulative,guilt_or_fear,fear_appeal
903,Medical,non_manipulative,non_manipulative,sympathy_exploitation
6321,Animals,non_manipulative,non_manipulative,sympathy_exploitation


## 3. Browse a few manipulative examples by category

In [5]:
for cat in ['sympathy_exploitation', 'artificial_urgency', 'guilt_framing', 'fear_appeal']:
    row = labelled[labelled[f'score_{cat}'].rank(ascending=False) == 1].iloc[0]
    print(f'=== highest {cat} score ===')
    print(row['text'][:400], '\n')


=== highest sympathy_exploitation score ===
For our supporters and those following our story, you know it has been a few very bad and stressful months. In and out of the hospital where they will only do enough to keep him alive instead of doing the surgeries he needs, however after talking to many locals and foriegners here, we know it is best not to have any surgery here however we can't leave because they are holding our ID'S until they g 

=== highest artificial_urgency score ===
UPDATE: After another week long stay in the hospital, we finally got some answers! It has been confirmed that her condition is indeed Dysautonomia. The doctors say we have a very long 10 years ahead of us. Up next will be more biopsy’s, more scans, more tests and we have to have meetings with the school to help accommodate all her new medical needs. While we are very encouraged by the progress we  

=== highest guilt_framing score ===
**VERSION EN ESPAÑOL MÁS ABAJO** Hi! My name is Mari and many of you (fam